In [1]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env = env.unwrapped

n_states = env.observation_space.n
n_actions = env.action_space.n

gamma = 0.99
theta = 1e-8

In [2]:
policy = np.ones((n_states, n_actions)) / n_actions


def policy_evaluation(env, policy, gamma=0.99, theta=1e-8):

    V = np.zeros(n_states)

    while True:

        delta = 0

        for s in range(n_states):

            v = V[s]
            value = 0

            for a in range(n_actions):

                action_prob = policy[s][a]

                for prob, next_state, reward, done in env.P[s][a]:
                    value += action_prob * prob * (
                        reward + gamma * V[next_state]
                    )

            V[s] = value
            delta = max(delta, abs(v - V[s]))

        if delta < theta:
            break

    return V

In [3]:
V = policy_evaluation(env, policy, gamma, theta)

print("Policy Evaluation - Value Function")
print("")
print(np.round(V.reshape(4,4),4))

Policy Evaluation - Value Function

[[0.0124 0.0104 0.0193 0.0095]
 [0.0148 0.     0.0389 0.    ]
 [0.0326 0.0843 0.1378 0.    ]
 [0.     0.1703 0.4336 0.    ]]


In [4]:
def policy_improvement(env, V, gamma=0.99):

    policy = np.zeros((n_states,n_actions))

    for s in range(n_states):

        action_values = np.zeros(n_actions)

        for a in range(n_actions):

            for prob,next_state,reward,done in env.P[s][a]:
                action_values[a] += prob*(reward+gamma*V[next_state])

        best_action=np.argmax(action_values)

        policy[s][best_action]=1

    return policy

policy = policy_improvement(env,V,gamma)

action_symbols={
    0:"←",
    1:"↓",
    2:"→",
    3:"↑"
}

best_actions=np.argmax(policy,axis=1)

policy_grid=np.array(
    [action_symbols[a] for a in best_actions]
).reshape(4,4)

print("Policy Improvement")
print("")
print(policy_grid)

Policy Improvement

[['←' '↑' '←' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]


In [7]:
def policy_iteration(env,policy,gamma=0.99,theta=1e-8):

    while True:

        V=policy_evaluation(env,policy,gamma,theta)

        new_policy=policy_improvement(env,V,gamma)

        if np.array_equal(policy,new_policy):
            break

        policy=new_policy

    return policy,V

optimal_policy,optimal_value_function=policy_iteration(
    env,
    policy,
    gamma,
    theta
)


In [8]:
# -------------------------------------------------
# Display Functions
# -------------------------------------------------

def print_value_function(V):
    print("\nOptimal State-Value Function:")
    print(np.round(V.reshape(4, 4), 4))


def print_policy(policy):

    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    best_actions = np.argmax(policy, axis=1)

    policy_grid = np.array(
        [action_symbols[action] for action in best_actions]
    ).reshape(4, 4)

    print("\nOptimal Policy:")
    print("")
    print(policy_grid)

In [9]:

# -------------------------------------------------
# Run Policy Iteration
# -------------------------------------------------

optimal_policy, optimal_value_function = policy_iteration(
    env,
    policy,
    gamma,
    theta
)

print("Name: AVINASH T")
print("Register Number: 212223230026")

print_value_function(optimal_value_function)
print_policy(optimal_policy)

env.close()

Name: AVINASH T
Register Number: 212223230026

Optimal State-Value Function:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
 [0.5918 0.6431 0.6152 0.    ]
 [0.     0.7417 0.8628 0.    ]]

Optimal Policy:

[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]
